## GraphFlow in Autogen

In [1]:
import os
from dotenv import load_dotenv
from autogen_ext.models.openai import OpenAIChatCompletionClient

## LLM
load_dotenv()
open_router_api_key = os.getenv('OPEN_ROUTER_API_KEY')

model_client =  OpenAIChatCompletionClient(
    base_url="https://openrouter.ai/api/v1",
    model="deepseek/deepseek-chat-v3.1:free",
    api_key = open_router_api_key,
    model_info={
        "family":'deepseek',
        "vision" :True,
        "function_calling":True,
        "json_output": False
    }
)

g:\AgenticAI-Autogen\autogen-venv\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:453: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [2]:
from autogen_agentchat.agents import AssistantAgent,UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

### Sequential Flow

In [3]:
writer = AssistantAgent(
    name="Writer",
    description="A writer agent that generates text based on user input.",
    model_client=model_client,
    system_message="You are a creative writer. Please write a story based on the user's input.",
)

reviewer = AssistantAgent(
    name="Reviewer",
    description="A reviewer agent that provides feedback on the text generated by the writer.",
    model_client=model_client,
    system_message="You are a reviewer. Please provide feedback on the text generated by the writer.",
)

In [4]:
builder = DiGraphBuilder()
builder.add_node(writer).add_node(reviewer)
builder.add_edge(writer, reviewer)

graph = builder.build()

In [5]:
graph

DiGraph(nodes={'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)

In [6]:
team = GraphFlow([writer,reviewer], graph)

In [7]:
stream = team.run_stream(task ="Write a good poem about India in less than 30 words.")

async for event in stream:
    print(event)

id='6ffc5c6e-4fa5-4b87-9601-fdebc32f1963' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 30, 3, 16, 37, 60220, tzinfo=datetime.timezone.utc) content='Write a good poem about India in less than 30 words.' type='TextMessage'
id='a4aa4a0f-fca4-445d-832a-1f2b2f21d54f' source='Writer' models_usage=RequestUsage(prompt_tokens=34, completion_tokens=34) metadata={} created_at=datetime.datetime(2025, 9, 30, 3, 16, 41, 878668, tzinfo=datetime.timezone.utc) content="Golden sun on ancient stone,\nSpice-scented air, a vibrant throng.\nWhere sacred rivers ever flow,\nIndia's heart beats strong and slow." type='TextMessage'
id='fd9f16cf-9ab5-42b1-9741-3804eb196159' source='Reviewer' models_usage=RequestUsage(prompt_tokens=65, completion_tokens=41) metadata={} created_at=datetime.datetime(2025, 9, 30, 3, 16, 43, 337484, tzinfo=datetime.timezone.utc) content='Your poem beautifully captures the essence of India with vivid imagery and economy of words. The rhythm is smoo